# KernelPack RAG — Retrieval Experiments (v1→v3)

Baseline hybrid retrieval pipeline on `kernelpack-python`. Each version
isolates one change, runs the same eval, and diagnoses failure modes before
moving to the next knob.

## Experiment Versions

| Version | Scope | Change | Hypothesis | recall@3 |
|---------|-------|--------|------------|----------|
| v1 | full index | baseline — raw source, MIN_LINES=5 | — | 5/10 (50%) |
| v2 | full index | MIN_LINES=1 | short functions are real targets | 4/10 (40%) |
| v3a | solvers only | no summaries (control) | smaller index = less noise | 5/10 (50%) |
| v3 | solvers only | LLM summaries prepended | summaries close the vocabulary gap | 8/10 (80%) |

## Key Findings

- All 10 targets land in the top-50 dense neighborhood after summaries — the bottleneck is ranking, not vocabulary gap.
- MIN_LINES=1 adds noise and hurts recall (5→4). There were no not-indexed failures in v1 to recover.
- A cross-encoder reranker is the highest-leverage next step.

## Contents

| Section | Description |
|---------|-------------|
| Setup | Deps, chunking, model, helpers, eval data |
| v1 — Baseline | Establish the recall@3 floor |
| v2 — MIN_LINES=1 | Recover short-function targets |
| v3a — Solvers-only | Isolate scoping effect (control) |
| v3 — LLM Summaries | Close the vocabulary gap |
| v3 — Results & Findings | Summary across all versions + next steps |
| Conclusions | What's next |


## Eval Set

**Source:** `qa_pairs/solvers_qa.json` — 10 pairs covering `kernelpack.solvers`.  
**Generator:** Codex, prompted to write questions a scientist would ask if they
knew what they wanted numerically but didn't know the KernelPack API.  
**Tiers:** api (4), workflow (3), conceptual (3).  
**Validation:** Codex-generated. Ground truth unconfirmed — treat recall numbers
as directional only until Varun reviews. Summaries used in v3 are also Codex-generated; conclusions should be treated as directional until Varun validates both QA pairs and summaries.

**Fixed across all versions:**
- tree-sitter AST chunking at `function_definition` boundaries
- UniXcoder dense embeddings (ChromaDB, L2)
- BM25 sparse leg, identifier-aware tokenization
- RRF fusion (k=60), top-3 returned
- Failure mode diagnosis via wide retrieval (top-50, dense-only) after each version

## Setup

In [1]:
%pip install -q \
    sentence-transformers==5.5.0 \
    tree-sitter==0.25.2 \
    tree-sitter-python==0.25.0 \
    chromadb==1.5.9 \
    rank-bm25==0.2.2


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import numpy as np
from pathlib import Path

import chromadb
import tree_sitter_python as tspython
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tree_sitter import Language, Parser

# ── paths ──────────────────────────────────────────────────────────────────────
# Assumes kernelpack-python is cloned in the same parent directory as this repo.
# If not, update this path to point at your local kernelpack-python/src/kernelpack.
REPO_PATH = Path("../../kernelpack-python/src/kernelpack")
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"kernelpack-python not found at {REPO_PATH.resolve()}. "
        "Clone it as a sibling directory: git clone https://github.com/ShankarLab/kernelpack-python"
    )

# Index built against kernelpack-python @ b0cc141 (Add Euler convergence study)
# Do NOT git pull without rebuilding the index and re-running eval
KERNELPACK_COMMIT = "b0cc141"

# ── retrieval config ───────────────────────────────────────────────────────────
TOP_K = 3   # results returned per query

/Users/jordanchambers/public-projects/scientific-codebase-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from utils.chunking import (
    load_docs, get_class_name, extract_chunks,
    bm25_tokenize, make_chunk_id, make_metadata,
)
from utils.eval_utils import (
    is_hit, run_eval, print_recall, eval_query,
    compare_versions, diagnose_wide, diagnose_wide_hybrid,
)

In [4]:
# Load all .py files from the repo
docs = load_docs(REPO_PATH)

Loaded 34 files from ../../kernelpack-python/src/kernelpack


In [5]:
# CHUNKING
MIN_LINES = 5   # functions with fewer than 5 lines dropped (inclusive)

# tree-sitter parser setup
PY_LANGUAGE = Language(tspython.language())
parser_ts   = Parser(PY_LANGUAGE)


all_chunks, all_dropped = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"])
    all_chunks.extend(kept)
    all_dropped.extend(dropped)

print(f"Chunks kept   : {len(all_chunks)}")
print(f"Chunks dropped: {len(all_dropped)}  (functions < {MIN_LINES} lines)")

Chunks kept   : 348
Chunks dropped: 277  (functions < 5 lines)


## Embedding Model & ChromaDB Client

Initialized once here and shared across all index versions.

In [6]:
model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

client = chromadb.EphemeralClient()
print("Embedding model and ChromaDB client ready.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14517.20it/s]


Embedding model and ChromaDB client ready.


## Helper Functions

Run this cell once. All experiment cells below depend on these functions.

| Function | Purpose |
|----------|---------|
| `make_chunk_id(chunk)` | Stable chunk ID: relative path + line range |
| `make_metadata(chunk)` | Standard metadata dict for ChromaDB storage |
| `build_index(chunks, name, documents)` | Build ChromaDB collection + BM25 index |
| `retrieve(query, index, k, silent)` | Hybrid BM25 + dense retrieval with RRF |
| `is_hit(result_meta, source_symbol)` | Symbol-match hit check |
| `run_eval(qa_pairs, index, k)` | Run recall@k; return (hits, totals) by tier |
| `print_recall(hits, totals, k, label)` | Print recall breakdown by tier |
| `eval_query(qa, index, k)` | Print per-query retrieval results + verdict |
| `compare_versions(qa_pairs, idx_a, idx_b, ...)` | Print hit/miss changes between versions |
| `diagnose_wide(qa_pairs, index, n)` | Dense-only top-n diagnostic (isolates embedding quality) |

In [7]:
# ── Index builder ──────────────────────────────────────────────────────────────

def build_index(
    chunks: list[dict],
    collection_name: str,
    documents: list[str] | None = None,
) -> dict:
    """Build a ChromaDB collection + BM25 index from a list of chunks.

    Args:
        chunks:          Output of extract_chunks.
        collection_name: Name for the ChromaDB collection.
        documents:       Pre-built document strings (e.g. summary-enriched text).
                         If None, raw chunk source text is used.
    Returns:
        dict with keys: col (ChromaDB collection), bm25 (BM25Okapi), ids (list[str]).
    """
    if documents is None:
        documents = [c["text"] for c in chunks]

    raw_ids   = [make_chunk_id(c) for c in chunks]
    metadatas = [make_metadata(c) for c in chunks]

    # deduplicate by ID
    seen = set()
    deduped_docs, deduped_metas, deduped_ids = [], [], []
    for doc, meta, cid in zip(documents, metadatas, raw_ids):
        if cid not in seen:
            seen.add(cid)
            deduped_docs.append(doc)
            deduped_metas.append(meta)
            deduped_ids.append(cid)

    col = client.get_or_create_collection(collection_name, embedding_function=ef)
    col.add(documents=deduped_docs, metadatas=deduped_metas, ids=deduped_ids)
    print(f"Collection {collection_name!r}: {col.count()} chunks")

    bm25 = BM25Okapi([bm25_tokenize(doc) for doc in deduped_docs])
    return {"col": col, "bm25": bm25, "ids": deduped_ids}


# ── Retrieval ──────────────────────────────────────────────────────────────────

def retrieve(query: str, index: dict, k: int = TOP_K, silent: bool = False) -> list[dict]:
    """Hybrid BM25 + dense retrieval with RRF fusion.

    Args:
        query:  Query string.
        index:  Dict with keys col, bm25, ids — output of build_index.
        k:      Number of results to return.
        silent: If True, suppresses printed output.
    Returns:
        Ordered list of metadata dicts for the top-k chunks.
    """
    col  = index["col"]
    bm25 = index["bm25"]
    ids  = index["ids"]

    # dense leg — embed with UniXcoder, return top 10 by L2 distance
    dense_res = col.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]
    dense_l2  = {cid: d for cid, d in zip(dense_res["ids"][0], dense_res["distances"][0])}

    # BM25 leg — scored by keyword overlap, return top 10
    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [ids[i] for i in top_bm25]

    # RRF fusion — score += 1/(rank + 60) for each list the chunk appears in
    # k=60 is the smoothing constant; dampens the advantage of being ranked 1st
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]

    # col.get() returns results in input-ID order — reorder to match RRF ranking
    fetched   = col.get(ids=top_ids, include=["documents", "metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    ordered_ids   = [cid for cid in top_ids if cid in id_to_idx]
    ordered_docs  = [fetched["documents"][id_to_idx[cid]] for cid in ordered_ids]
    ordered_metas = [fetched["metadatas"][id_to_idx[cid]] for cid in ordered_ids]

    if not silent:
        print(f'\nQUERY: "{query}"')
        print("=" * 60)
        for i, (cid, doc, meta) in enumerate(zip(ordered_ids, ordered_docs, ordered_metas), 1):
            cls    = f"{meta['class_name']}." if meta["class_name"] else ""
            l2_str = f"{dense_l2[cid]:.4f}" if cid in dense_l2 else "n/a (BM25-only)"
            print(f"\n  Rank {i} | RRF {rrf[cid]:.4f} | L2 {l2_str}")
            print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
            print("  " + "-" * 56)
            print("  " + doc[:300].replace("\n", "\n  "))

    return ordered_metas

def retrieve_wide(query: str, index: dict, n: int = 50) -> list[dict]:
    """Hybrid BM25 + dense with expanded candidate pool. Used by diagnose_wide_hybrid."""
    col  = index["col"]
    bm25 = index["bm25"]
    ids  = index["ids"]

    dense_res = col.query(query_texts=[query], n_results=n)
    dense_ids = dense_res["ids"][0]

    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:n]
    bm25_ids    = [ids[i] for i in top_bm25]

    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:n]
    fetched   = col.get(ids=top_ids, include=["metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    return [fetched["metadatas"][id_to_idx[cid]] for cid in top_ids if cid in id_to_idx]

print("Helper functions defined.")

Helper functions defined.


## Load Eval Data

QA pairs are loaded once here and reused across all versions.

In [8]:
with open("qa_pairs/solvers_qa.json") as f:
    qa_pairs_solvers = json.load(f)

with open("qa_pairs/benchmark_qa.json") as f:
    qa_pairs_benchmark = json.load(f)

print(f"Solvers QA pairs  : {len(qa_pairs_solvers)}")
print(f"Benchmark QA pairs: {len(qa_pairs_benchmark)}")

Solvers QA pairs  : 10
Benchmark QA pairs: 3


## v1 — Baseline

**Index state:**
- MIN_LINES=5 — functions shorter than 5 lines dropped
- 337 chunks indexed, raw source text only
- No plain-English descriptions — queries are natural language, chunks are raw code


In [9]:
idx_v1 = build_index(all_chunks, "kp-hybrid-v1")

Collection 'kp-hybrid-v1': 348 chunks


In [10]:
for qa in qa_pairs_solvers:
    eval_query(qa, idx_v1)


QUERY : For a publication benchmark with mixed Dirichlet and Neumann elliptic boundary data, how can I obtain the physical field together with the assembled matrix, right-hand side pieces, and nullspace metadata needed for reproducibility?
TIER  : api
TARGET: PoissonSolver.solve (src/kernelpack/solvers/poisson.py)
ANSWER: The stationary Poisson API materializes the Neumann and Dirichlet boundary coefficients, assembles the boundary operator for those coefficients, evaluates the forcing on interior-plus-boundary nodes, and evaluates the boundary right-hand side. It treats the case as pure Neumann when the maximum absolute Dirichlet coefficient is at most 1e-13, augments the sparse system with a constant-mode constraint in that case, and uses a direct sparse solve unless an initial guess triggers GMRES with fallback. The result dictionary contains `u`, `full_state`, `L`, `BC`, `system_matrix`, `rhs`, `target_rhs`, `boundary_rhs`, `used_nullspace_augmentation`, and `lagrange_multiplier`.

In [11]:
hits_v1, totals_v1 = run_eval(qa_pairs_solvers, idx_v1)
print_recall(hits_v1, totals_v1, k=TOP_K, label="v1 — solvers")

── Recall@3 by tier — v1 — solvers
  api          2/4  (50%)
  workflow     2/3  (67%)
  conceptual   1/3  (33%)
  overall      5/10  (50%)


In [12]:
for qa in qa_pairs_benchmark:
    eval_query(qa, idx_v1)

hits_bench_v1, totals_bench_v1 = run_eval(qa_pairs_benchmark, idx_v1)
print_recall(hits_bench_v1, totals_bench_v1, k=TOP_K, label="v1 — benchmark")


QUERY : how to create an RBF-FD differentiation matrix?
TIER  : workflow
TARGET: FDDiffOp.assemble_op (src/kernelpack/rbffd/core.py)
ANSWER: Use the RBF-FD classes in `kernelpack.rbffd`. Create stencil settings with `StencilProperties.from_accuracy(operator=..., convergence_order=..., dimension=..., approximation="rbf", tree_mode="all", point_set="interior_boundary")`, create operator settings with `OpProperties(...)`, then instantiate `FDDiffOp(lambda: RBFStencil())`. Call `assemble_op(domain, op_name, st_props, op_props, neu_coeff=None, dir_coeff=None, active_rows=None)`, where `op_name` is a string such as `"lap"`, `"grad"`, or `"bc"` as supported by the stencil code. Retrieve the sparse CSR differentiation matrix with `get_op()`. For overlapped assembly, the source also provides `FDODiffOp` with the same `assemble_op` and `get_op` workflow.
──────────────────────────────────────────────────────────────────────
  Rank 1: solvers/_common.py — make_assembler
  Rank 2: divfree/core.py

In [13]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v1)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v1)

── dense ──
  PoissonSolver.solve                                           dense: 18          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 43          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 47          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 3           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 1           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 4           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 2           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 2           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 29          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v1 — Result

recall@3: **5/10 (50%)**

| Tier | Hits |
|------|------|
| api | 2/4 |
| workflow | 2/3 |
| conceptual | 1/3 |

### qa_solvers.json — k@3, 5, 10
| k | recall (hybrid) |
|---|-----------------|
| 3 | 5/10 |
| 5 | 7/10 |
| 10 | 8/10 |

All 10 targets found in top-50 dense. Every miss is a ranking problem — zero not-indexed failures.


## v1 — Failure Mode Analysis

Ran wide retrieval (top-50) on all 10 eval pairs to classify each miss.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: corrected prior discrepancies before rebuilding this table: `VariablePoissonSolver.solve` v1 dense is 41 in the executed `diagnose_wide` output, and `NonlinearVariablePoissonSolver.solve` v1 hybrid is 6 in the executed `diagnose_wide_hybrid` output.

| Target | v1 dense | v1 hybrid | Category |
|---|---|---|---|
| PoissonSolver.solve | 18 | 2 | Reranker |
| VariablePoissonSolver.solve | 41 | 8 | Reranker problem |
| NonlinearVariablePoissonSolver.solve | 47 | 6 | Reranker problem |
| PUSLAdvectionSolver.forward_sl_step | 3 | 2 | Hit |
| DiffusionSolver.set_state_history | 1 | 1 | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 4 | Reranker problem |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 1 | Hit |
| build_stencil_properties | 2 | 1 | Hit |
| pu_patch_weight | 29 | 13 | Reranker problem |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 5 | Reranker problem |

### Key finding
- 5 targets reach hybrid rank 1-3 in the wide diagnostic.
- 5 remaining targets are ranking failures: the target appears in the wide diagnostic but does not reach hybrid rank 1-3.
- 0 not-indexed failures: every target appears in the executed top-50 diagnostics.

All misses visible in this table are ranking failures, not retrieval failures.
A cross-encoder that re-scores hybrid candidates by reading query+chunk together
addresses the problem directly.

## Tuning Plan

### Confirmed failure modes (from v1 wide diagnostic)

All 5 misses are **reranker problems** — the target chunk exists and is referenced
in the top-10 neighborhood but doesn't surface in the hybrid top-3:

| Target | Dense rank@50 | Failure type |
|--------|--------------|--------------|
| PoissonSolver.solve | 18 | Reranker — buried |
| VariablePoissonSolver.solve | 41 | Reranker — buried |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | Reranker — BM25 drag |
| pu_patch_weight | 29 | Reranker — buried |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | Reranker — buried |

### Experiment sequence

| Version | Change | Failure mode targeted | Status |
|---------|--------|-----------------------|--------|
| v2 | MIN_LINES=1 | recover any not-indexed targets | done — hurt recall (5→4), no not-indexed failures existed |
| v3a | solvers-only scope (control) | isolate index noise effect | done — 5/10, same as v1 |
| v3 | LLM summaries | vocabulary gap / dense ranking | done — 8/10 |
| v4 (next) | cross-encoder reranker | 2 remaining reranker misses | pending |


## v2 — MIN_LINES=1

**Change:** Drop minimum line filter from 5 to 1 

In [14]:
all_chunks_v2, all_dropped_v2 = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"], min_lines=1)
    all_chunks_v2.extend(kept)
    all_dropped_v2.extend(dropped)

print(f"v1 chunks: {len(all_chunks)}")
print(f"v2 chunks: {len(all_chunks_v2)}")

idx_v2 = build_index(all_chunks_v2, "kp-hybrid-v2")


v1 chunks: 348
v2 chunks: 625
Collection 'kp-hybrid-v2': 625 chunks


In [15]:
hits_v2, totals_v2 = run_eval(qa_pairs_solvers, idx_v2)
print_recall(hits_v2, totals_v2, k=TOP_K, label="v2 — solvers")

print("── v1 vs v2 recall@3 ─────────────────")
print(f"  v1 overall: {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall: {sum(hits_v2.values())}/10 ({round(100 * sum(hits_v2.values()) / 10)}%)")

── Recall@3 by tier — v2 — solvers
  api          1/4  (25%)
  workflow     2/3  (67%)
  conceptual   1/3  (33%)
  overall      4/10  (40%)
── v1 vs v2 recall@3 ─────────────────
  v1 overall: 5/10 (50%)
  v2 overall: 4/10 (40%)


In [16]:
compare_versions(qa_pairs_solvers, idx_v1, idx_v2, "v1", "v2")

✓ -> ✗ REGRESSED: NonlinearVariablePoissonSolver.solve


In [17]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v2)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v2)

── dense ──
  PoissonSolver.solve                                           dense: 33          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: not found   mentioned-in-top-10: ✗
  NonlinearVariablePoissonSolver.solve                          dense: not found   mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 5           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 7           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 7           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 7           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 2           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 42          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

In [18]:
# what does the added noise look like?
new_chunks = [c for c in all_chunks_v2 if (c["end_line"] - c["start_line"] + 1) < 5]
for c in new_chunks[:20]:
    lines = c["end_line"] - c["start_line"] + 1
    print(f"{lines} lines | {c['class_name']}.{c['function_name']}")
    print("  " + c["text"][:120].replace("\n", "\n  "))
    print()

3 lines | None._stack_field
  def _stack_field(U: np.ndarray) -> np.ndarray:
      U = np.asarray(U, dtype=float)
      return np.concatenate([U[:, d] for

4 lines | None._unstack_field
  def _unstack_field(rhs: np.ndarray, dim: int) -> np.ndarray:
      rhs = np.asarray(rhs, dtype=float).reshape(-1)
      n = 

4 lines | LocalDivFreeInterpolator._query_stencil_indices
  def _query_stencil_indices(self, point: np.ndarray) -> np.ndarray:
          idx = self.Tree.query(point, k=self.StencilSi

4 lines | LocalDivFreeInterpolator.assign_centers
  def assign_centers(self, Xq: np.ndarray) -> np.ndarray:
          Xq = np.asarray(Xq, dtype=float)
          idx = self.Tree

2 lines | DomainDescriptor.set_outer_level_set
  def set_outer_level_set(self, level_set: object) -> None:
          self.outer_level_set = level_set

2 lines | DomainDescriptor.set_sep_rad
  def set_sep_rad(self, sep_rad: float) -> None:
          self.sep_rad = float(sep_rad)

2 lines | DomainDescriptor.set_boundary_leve

In [19]:
# v2 benchmark eval
hits_bench_v2, totals_bench_v2 = run_eval(qa_pairs_benchmark, idx_v2)

n = len(qa_pairs_benchmark)
print("── Benchmark recall@3 ────────────────")
print(f"  v1: {sum(hits_bench_v1.values())}/{n}")
print(f"  v2: {sum(hits_bench_v2.values())}/{n}")
print("  v3: n/a — solvers-only index, benchmark comparison not valid")

── Benchmark recall@3 ────────────────
  v1: 0/3
  v2: 0/3
  v3: n/a — solvers-only index, benchmark comparison not valid


## v2 — Result

recall@3: **4/10 (40%)** — down from 5/10. `NonlinearVariablePoissonSolver.solve`
regressed from hit (hybrid rank 2 in v1) to miss.

Adding short functions (337→614 chunks) introduced index noise that degraded ranks
for most previously-ranked targets. MIN_LINES is a poor proxy filter — there were
no not-indexed misses to recover in v1.

## v2 — Failure Mode Analysis

Wide retrieval (top-50), compared against v1 baseline.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: corrected the inherited v1 dense value for `VariablePoissonSolver.solve` from 40 to 41 to match the executed v1 `diagnose_wide` output. All v2 dense and hybrid ranks below match the executed v2 diagnostic outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | Category |
|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 2 | 33 | 3 | Recovered by hybrid |
| VariablePoissonSolver.solve | 41 | 8 | not found | 27 | **Hard regression** |
| NonlinearVariablePoissonSolver.solve | 47 | 6 | not found | 10 | Regressed |
| PUSLAdvectionSolver.forward_sl_step | 3 | 2 | 5 | 2 | Stable hit |
| DiffusionSolver.set_state_history | 1 | 1 | 7 | 1 | Stable hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 4 | 7 | 5 | Reranker problem |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 1 | 7 | 1 | Stable hit |
| build_stencil_properties | 2 | 1 | 2 | 1 | Stable hit |
| pu_patch_weight | 29 | 13 | 42 | 14 | Reranker problem |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 5 | 19 | 5 | Reranker problem |

- `VariablePoissonSolver.solve` is the only target explicitly reported as `not found` in the v2 dense output.
- BM25 still recovers several dense degradations into the hybrid top ranks, but `VariablePoissonSolver.solve`, `NonlinearVariablePoissonSolver.solve`, `MultiSpeciesDiffusionSolver.set_initial_state`, `pu_patch_weight`, and `IncompressibleEulerBDFBackend._get_cached_system` remain outside hybrid rank 1-3.
- No v1 not-indexed failures existed to recover, so MIN_LINES=1 had no upside in this diagnostic table.

## v3a — Solvers-only (Control)

**Change:** Scope index to `kernelpack.solvers` only — no summaries.  
**Hypothesis:** Smaller, focused index reduces noise independently of summary enrichment.  
**Purpose:** Isolates scoping effect so v3's summary contribution can be measured cleanly.

In [20]:
# solvers-only chunk list — shared by v3a and v3
all_chunks_v3 = [c for c in all_chunks if "solvers" in c["path"]]
print(f"Solvers chunks: {len(all_chunks_v3)}")

Solvers chunks: 187


In [21]:
# v3a — solvers-only index, no summaries
# control experiment: isolates whether scoping to solvers alone improves recall
# independent of summary enrichment
idx_v3a = build_index(all_chunks_v3, "kp-hybrid-v3a")

hits_v3a, totals_v3a = run_eval(qa_pairs_solvers, idx_v3a)
print_recall(hits_v3a, totals_v3a, k=TOP_K, label="v3a — solvers")

print("── v1 vs v3a recall@3 ────────────────")
print(f"  v1  (full index, no summaries)   : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries) : {sum(hits_v3a.values())}/10")

Collection 'kp-hybrid-v3a': 187 chunks
── Recall@3 by tier — v3a — solvers
  api          2/4  (50%)
  workflow     2/3  (67%)
  conceptual   1/3  (33%)
  overall      5/10  (50%)
── v1 vs v3a recall@3 ────────────────
  v1  (full index, no summaries)   : 5/10
  v3a (solvers only, no summaries) : 5/10


In [22]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v3a)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v3a)

── dense ──
  PoissonSolver.solve                                           dense: 15          mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 30          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 40          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 2           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 1           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 4           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 2           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 1           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 16          mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v3a — Failure Mode Analysis

recall@3: **5/10 (50%)** — same as v1. Scoping to solvers alone did not improve
recall over the full-index baseline.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: no discrepancies found between the v3a dense/hybrid columns below and the executed v3a diagnostic outputs. The v1/v2 carry-forward columns also match their executed diagnostic outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | v3a dense | v3a hybrid | Category |
|---|---|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 2 | 33 | 3 | 15 | 2 | Hit |
| VariablePoissonSolver.solve | 41 | 8 | not found | 27 | 30 | 11 | Reranker problem |
| NonlinearVariablePoissonSolver.solve | 47 | 6 | not found | 10 | 40 | 6 | Reranker problem |
| PUSLAdvectionSolver.forward_sl_step | 3 | 2 | 5 | 2 | 2 | 2 | Hit |
| DiffusionSolver.set_state_history | 1 | 1 | 7 | 1 | 1 | 1 | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 4 | 7 | 5 | 4 | 4 | Reranker problem |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 1 | 7 | 1 | 2 | 1 | Hit |
| build_stencil_properties | 2 | 1 | 2 | 1 | 1 | 1 | Hit |
| pu_patch_weight | 29 | 13 | 42 | 14 | 16 | 9 | Reranker problem |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 5 | 19 | 5 | 14 | 4 | Reranker problem |

Confirms the v3 gain is attributable to summaries, not the smaller index.

## v3 — LLM Summaries

**Change:** Prepend Codex-generated plain-English summary to each solver chunk.  
**Hypothesis:** Bridging the vocabulary gap between NL queries and code identifiers recovers buried targets.

In [23]:
with open("metadata/solvers_summaries.json") as f:
    summaries = json.load(f)

# key = (function_name, class_name)
summary_lookup = {
    (s["function_name"], s["class_name"] or ""): s["summary"]
    for s in summaries
}
print(f"Loaded {len(summary_lookup)} summaries")

Loaded 366 summaries


In [24]:
# build enriched documents for v3 — prepend LLM summary to each chunk
documents_v3 = []
for c in all_chunks_v3:
    key     = (c["function_name"], c["class_name"] or "")
    summary = summary_lookup.get(key, "")
    enriched = f"{summary}\n\n{c['text']}" if summary else c["text"]
    documents_v3.append(enriched)

matched_v3 = sum(1 for c in all_chunks_v3
                 if (c["function_name"], c["class_name"] or "") in summary_lookup)
print(f"{matched_v3}/{len(all_chunks_v3)} chunks matched a summary")

187/187 chunks matched a summary


In [25]:
idx_v3 = build_index(all_chunks_v3, "kp-hybrid-v3", documents=documents_v3)

# sanity check: confirm summary is prepended to raw source
sample = idx_v3["col"].get(limit=1, include=["documents"])
print("\n── Sample enriched document (first 500 chars) ──")
print(sample["documents"][0][:500])

Collection 'kp-hybrid-v3': 187 chunks

── Sample enriched document (first 500 chars) ──
Translates a solver accuracy/order request and domain dimension into the stencil size, polynomial degree, spline degree, tree mode, and point set used by the RBF-FD assembler.

def build_stencil_properties(domain: DomainDescriptor, xi: int, theta: int, point_set: str) -> StencilProperties:
    # The solver layer asks for target order `xi` and operator order `theta`;
    # here we translate that request into a concrete stencil size and
    # polynomial degree that the rbffd layer understands.
   


In [26]:
hits_v3, totals_v3 = run_eval(qa_pairs_solvers, idx_v3)
print_recall(hits_v3, totals_v3, k=TOP_K, label="v3 — solvers")

print("── v1 vs v3a vs v3 recall@3 ──────────")
print(f"  v1  (full index, no summaries)     : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries)   : {sum(hits_v3a.values())}/10")
print(f"  v3  (solvers only, with summaries) : {sum(hits_v3.values())}/10")

── Recall@3 by tier — v3 — solvers
  api          2/4  (50%)
  workflow     3/3  (100%)
  conceptual   3/3  (100%)
  overall      8/10  (80%)
── v1 vs v3a vs v3 recall@3 ──────────
  v1  (full index, no summaries)     : 5/10
  v3a (solvers only, no summaries)   : 5/10
  v3  (solvers only, with summaries) : 8/10


In [27]:
compare_versions(qa_pairs_solvers, idx_v1, idx_v3, "v1", "v3")

✗ -> ✓ RECOVERED: PoissonSolver.solve
✓ -> ✗ REGRESSED: NonlinearVariablePoissonSolver.solve
✗ -> ✓ RECOVERED: MultiSpeciesDiffusionSolver.set_initial_state
✗ -> ✓ RECOVERED: pu_patch_weight
✗ -> ✓ RECOVERED: IncompressibleEulerBDFBackend._get_cached_system


In [28]:
# Wide retrieval diagnostic (top-50): dense-only then hybrid.
# dense rank: embedding quality in isolation.
# hybrid rank: BM25 + RRF on top — delta vs dense isolates BM25 contribution.
# mentioned-in-top-10: target symbol appears in a retrieved chunk body (context reachability).
print("── dense ──")
diagnose_wide(qa_pairs_solvers, idx_v3)
print("\n── hybrid ──")
diagnose_wide_hybrid(qa_pairs_solvers, idx_v3)

── dense ──
  PoissonSolver.solve                                           dense: 6           mentioned-in-top-10: ✓
  VariablePoissonSolver.solve                                   dense: 18          mentioned-in-top-10: ✓
  NonlinearVariablePoissonSolver.solve                          dense: 18          mentioned-in-top-10: ✓
  PUSLAdvectionSolver.forward_sl_step                           dense: 4           mentioned-in-top-10: ✓
  DiffusionSolver.set_state_history                             dense: 4           mentioned-in-top-10: ✓
  MultiSpeciesDiffusionSolver.set_initial_state                 dense: 2           mentioned-in-top-10: ✓
  PUSLFDAdvectionDiffusionSolver.bdf3_step                      dense: 6           mentioned-in-top-10: ✓
  build_stencil_properties                                      dense: 1           mentioned-in-top-10: ✓
  pu_patch_weight                                               dense: 1           mentioned-in-top-10: ✓
  IncompressibleEulerBDFBackend._g

## v3 — Failure Mode Analysis

Wide retrieval (top-50), compared against v1, v2, and v3a baselines.

Note: Hybrid ranks in this table use expanded wide retrieval (top-50 dense + top-50 BM25) and are not directly comparable to recall@3, which uses top-10 dense + top-10 BM25.

> Output scan comment: corrected the inherited v1 dense value for `VariablePoissonSolver.solve` from 40 to 41 to match the executed v1 `diagnose_wide` output. All v3 dense and hybrid ranks below match the executed v3 diagnostic outputs.

| Target | v1 dense | v1 hybrid | v2 dense | v2 hybrid | v3a dense | v3a hybrid | v3 dense | v3 hybrid | Category |
|---|---|---|---|---|---|---|---|---|---|
| PoissonSolver.solve | 18 | 2 | 33 | 3 | 15 | 2 | 6 | 2 | Recovered hit |
| VariablePoissonSolver.solve | 41 | 8 | not found | 27 | 30 | 11 | 18 | 6 | Reranker problem |
| NonlinearVariablePoissonSolver.solve | 47 | 6 | not found | 10 | 40 | 6 | 18 | 3 | Wide diagnostic hit; eval regression |
| PUSLAdvectionSolver.forward_sl_step | 3 | 2 | 5 | 2 | 2 | 2 | 4 | 1 | Hit |
| DiffusionSolver.set_state_history | 1 | 1 | 7 | 1 | 1 | 1 | 4 | 3 | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 4 | 7 | 5 | 4 | 4 | 2 | 2 | Recovered hit |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 1 | 7 | 1 | 2 | 1 | 6 | 3 | Hit |
| build_stencil_properties | 2 | 1 | 2 | 1 | 1 | 1 | 1 | 1 | Hit |
| pu_patch_weight | 29 | 13 | 42 | 14 | 16 | 9 | 1 | 1 | Recovered hit |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 5 | 19 | 5 | 14 | 4 | 7 | 3 | Recovered hit |

- Summaries improve the wide diagnostic sharply for `pu_patch_weight`, `VariablePoissonSolver.solve`, and `NonlinearVariablePoissonSolver.solve` compared with v2.
- `VariablePoissonSolver.solve` remains the main ranking problem in v3: present in both dense and hybrid diagnostics, but still outside hybrid rank 1-3.
- `NonlinearVariablePoissonSolver.solve` is hybrid rank 3 in the wide diagnostic because that diagnostic uses an expanded top-50 candidate pool, so it does not contradict a recall@3 miss under the top-10 dense + top-10 BM25 eval path.

In [29]:
## Multi-k Recall Sweep
print(f"── v1 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v1, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v2 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v2, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v3a recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v3a, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

print(f"── v3 recall@3, 5, 10 ──────────")
for k in [3, 5, 10]:
    hits, totals = run_eval(qa_pairs_solvers, idx_v3, k=k)
    total_hits = sum(hits.values())
    total_pairs = sum(totals.values())
    print(f"k = {k}: {total_hits}/{total_pairs}")

── v1 recall@3, 5, 10 ──────────
k = 3: 5/10
k = 5: 7/10
k = 10: 8/10
── v2 recall@3, 5, 10 ──────────
k = 3: 4/10
k = 5: 6/10
k = 10: 7/10
── v3a recall@3, 5, 10 ──────────
k = 3: 5/10
k = 5: 6/10
k = 10: 9/10
── v3 recall@3, 5, 10 ──────────
k = 3: 8/10
k = 5: 10/10
k = 10: 10/10


## v3 — Results & Findings

### Recall@3
| Version | Change | recall@3 |
|---------|--------|----------|
| v1  | full index, no summaries (baseline) | 5/10 (50%) |
| v2  | MIN_LINES=1 | 4/10 (40%) |
| v3a | solvers only, no summaries | 5/10 (50%) |
| v3  | solvers only, with summaries | 8/10 (80%) |

LLM summaries added +3 over v1. Scoping alone contributed nothing.

### The real finding
All 10 targets are in the top-50 dense neighborhood in v3. The vocabulary gap is closed.
At recall@5 and recall@10, v3 hits 10/10 — a reranker window of top-5 is enough to
recover everything.

### Recoveries vs regressions (v1 → v3)
- **Recovered (✗→✓):** PoissonSolver.solve, MultiSpeciesDiffusionSolver.set_initial_state,
  pu_patch_weight, IncompressibleEulerBDFBackend._get_cached_system
- **Regressed (✓→✗):** NonlinearVariablePoissonSolver.solve — was hybrid rank 2 in v1
  (BM25 carrying it despite dense rank 47), now wide rank 18 and a miss in v3. The summary
  may have degraded its embedding, or BM25 signal weakened under the solvers-only scope.
  Needs investigation before v4.


# Conclusions

### What's still broken
- **Reranker problems (2):** VariablePoissonSolver.solve (wide dense rank 18) and
  NonlinearVariablePoissonSolver.solve (wide dense rank 18) — both outside recall@3
  hybrid top-3. Note: the wide top-50 diagnostic shows NonlinearVariablePoissonSolver.solve
  at hybrid rank 3, but this uses an expanded candidate pool and does not reflect the
  actual recall@3 eval path.
- **Open regression:** NonlinearVariablePoissonSolver.solve was a recall@3 hit in v1
  (BM25 lifted it despite dense rank 47). In v3 it is a miss. Root cause unclear —
  investigate before implementing v4.

### Next
Cross-encoder reranker. All 10 targets are in the top-50 dense window in v3.
A reranker window of top-5 suffices to recover 10/10 (v3 recall@5 = 10/10).
Implement `cross-encoder/ms-marco-MiniLM-L-6-v2` on the v3 index first,
then measure the delta on the same 10-query eval.
